# 中国地方政府债券利差高级计量经济学框架
## Advanced Econometric Framework for China Local Government Bond Spread Analysis

---

**作者**: 宏观对冲基金量化研究主管  
**核心理念**: 模型锦标赛 (Model Tournament) - 不依赖单一模型，让数据选择最优动态特征

---

### 分析框架三大支柱:

1. **波动率建模锦标赛** (GARCH/EGARCH/GJR-GARCH) - 捕捉不对称效应
2. **卡尔曼滤波器** - 从噪音中提取真实信号
3. **极值理论** (EVT) - 尾部风险量化

---

In [ ]:
# 环境配置与核心库导入
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# 计量经济学核心库
from arch import arch_model  # GARCH 模型族
from statsmodels.tsa.statespace.sarimax import SARIMAX  # Kalman Filter 的便捷实现
from scipy import stats
from scipy.optimize import minimize

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 所有依赖库加载完成")

---
## 第一部分: 灵活数据引擎 (Flexible Data Engine)
---

In [ ]:
# ============================================================================
# 全局配置字典 - 生产环境请切换到 'WIND_EDB'
# ============================================================================
CONFIG = {
    'SOURCE': 'MOCK',  # 'WIND_EDB' 或 'MOCK'
    'TICKER': 'M0017142',  # Wind EDB 中的地方债利差 Ticker (示例)
    'START_DATE': '2018-01-01',
    'END_DATE': '2025-12-31',
    'MAD_THRESHOLD': 5.0,  # MAD 倍数，超过视为异常值（Wind 数据经常有离群点）
    'GARCH_P': 1,  # GARCH 的 p 阶数
    'GARCH_Q': 1,  # GARCH 的 q 阶数
    'VaR_CONFIDENCE': 0.99,  # 99% VaR
    'EVT_THRESHOLD_PERCENTILE': 0.95  # EVT 的阈值选取（取超过 95% 分位的极值）
}

print(f"当前数据源: {CONFIG['SOURCE']}")
print(f"分析周期: {CONFIG['START_DATE']} 至 {CONFIG['END_DATE']}")

In [ ]:
class DataEngine:
    """
    数据引擎类 - 支持 Wind EDB 和 Mock 数据
    
    关键功能:
    1. 自动处理 Wind EDB 的假期数据缺失
    2. MAD 方法剔除极端异常值（比 Z-Score 更稳健）
    3. 统一输出格式：DatetimeIndex + 'spread' 列
    """
    
    def __init__(self, config):
        self.config = config
        self.raw_data = None
        self.clean_data = None
        
    def load_data(self):
        """加载数据 - 生产环境需接入 Wind API"""
        if self.config['SOURCE'] == 'WIND_EDB':
            # 生产环境代码示例（需要 Wind Python API）:
            # from WindPy import w
            # w.start()
            # data = w.edb(self.config['TICKER'], self.config['START_DATE'], self.config['END_DATE'])
            # self.raw_data = pd.DataFrame(data.Data, index=data.Times, columns=['spread'])
            raise NotImplementedError("请在生产环境配置 Wind API")
        
        else:  # MOCK 数据
            # 模拟一个真实的地方债利差时间序列：
            # - 均值回归特征 (Mean Reversion)
            # - 波动率聚集 (Volatility Clustering)
            # - 偶尔的尖峰 (Fat Tails)
            np.random.seed(42)
            dates = pd.date_range(self.config['START_DATE'], self.config['END_DATE'], freq='B')
            n = len(dates)
            
            # 使用 AR(1) + GARCH(1,1) 过程生成模拟数据
            spread = np.zeros(n)
            spread[0] = 100  # 初始利差 100 bps
            volatility = np.zeros(n)
            volatility[0] = 10
            
            # 这些参数是根据实际地方债市场经验设定的：
            # - phi = 0.98: 高度持久性（地方债利差是慢变量）
            # - omega 很小：长期波动率稳定
            # - alpha + beta 接近 1：波动率聚集效应明显
            phi = 0.98  # AR(1) 系数 - 高持久性
            mu = 100    # 长期均值 100 bps
            omega = 0.5 # GARCH 常数项
            alpha = 0.15 # ARCH 效应
            beta = 0.80  # GARCH 效应
            
            for t in range(1, n):
                # GARCH(1,1) 波动率更新
                volatility[t] = np.sqrt(omega + alpha * (spread[t-1] - mu)**2 + beta * volatility[t-1]**2)
                
                # AR(1) 过程 + 随机冲击
                shock = np.random.standard_t(df=5) * volatility[t]  # 使用 t 分布制造肥尾
                spread[t] = mu + phi * (spread[t-1] - mu) + shock
                
                # 偶尔添加跳跃（模拟政策冲击或信用事件）
                if np.random.rand() < 0.01:  # 1% 概率发生跳跃
                    spread[t] += np.random.choice([-1, 1]) * np.random.uniform(20, 40)
            
            self.raw_data = pd.DataFrame({'spread': spread}, index=dates)
            print(f"✓ 已生成 {len(dates)} 个交易日的模拟数据")
        
        return self.raw_data
    
    def clean_data(self):
        """
        数据清洗 - 使用 MAD (Median Absolute Deviation) 处理异常值
        
        为什么用 MAD 而不是标准差？
        - Wind EDB 的数据经常有「硬伤」：比如某天突然出现 999 或 -999 的占位符
        - 标准差对这种极端值敏感，会误判正常波动
        - MAD 基于中位数，对离群点有抵抗力（Robust Estimator）
        """
        if self.raw_data is None:
            raise ValueError("请先调用 load_data()")
        
        df = self.raw_data.copy()
        
        # Step 1: 处理缺失值（Wind 在节假日会返回 NaN）
        # 这里用 forward fill 是因为利差是慢变量，昨天的值是今天的最佳估计
        df['spread'] = df['spread'].fillna(method='ffill').fillna(method='bfill')
        
        # Step 2: MAD 异常值检测
        median = df['spread'].median()
        mad = np.median(np.abs(df['spread'] - median))
        
        # 这个 1.4826 是什么鬼？
        # 答：它是让 MAD 在正态分布下等价于标准差的调整因子
        # 但我们的数据不是正态分布（有肥尾），所以这只是个近似
        threshold = self.config['MAD_THRESHOLD']
        modified_z_score = 0.6745 * (df['spread'] - median) / mad
        
        outliers = np.abs(modified_z_score) > threshold
        if outliers.sum() > 0:
            print(f"⚠️  检测到 {outliers.sum()} 个异常值（MAD 阈值 = {threshold}）")
            # 用中位数替换异常值（保守做法，避免引入偏差）
            df.loc[outliers, 'spread'] = median
        
        self.clean_data = df
        print(f"✓ 数据清洗完成，最终样本量: {len(df)}")
        return self.clean_data
    
    def get_returns(self):
        """计算利差变化（一阶差分）- GARCH 模型的输入"""
        if self.clean_data is None:
            raise ValueError("请先调用 clean_data()")
        
        # 为什么用差分而不是百分比收益率？
        # 因为利差本身就是绝对值（bps），不是价格
        # 100 bps -> 105 bps 的波动意义和 200 bps -> 205 bps 一样
        returns = self.clean_data['spread'].diff().dropna()
        return returns

In [ ]:
# 执行数据加载与清洗
engine = DataEngine(CONFIG)
raw_data = engine.load_data()
clean_data = engine.clean_data()
returns = engine.get_returns()

print(f"\n数据统计摘要:")
print(clean_data['spread'].describe())
print(f"\n利差变化统计:")
print(returns.describe())

---
## 第二部分: 模块 A - 波动率建模锦标赛 (GARCH Tournament)
---

### 为什么需要三个模型竞争？

金融市场有个经验规律：**坏消息比好消息更容易引发波动** (Leverage Effect)

- **GARCH(1,1)**: 假设好坏消息对波动率的影响对称 → 通常不符合现实
- **EGARCH**: 用指数形式捕捉非对称效应 → 理论优雅但参数难估计
- **GJR-GARCH**: 用阈值函数区分正负冲击 → 实践中最稳定

我们让 AIC/BIC 来裁判谁更适合中国地方债市场。


In [ ]:
class VolatilityModeler:
    """
    波动率建模类 - 实现 GARCH 锦标赛
    
    技术细节:
    - 使用 Student-t 分布而非正态分布（地方债收益率有肥尾）
    - 设置较宽松的收敛容差（tol=1e-4），避免优化器在极端行情下卡住
    - 自动保存每个模型的 AIC/BIC，最后选出 Winner
    """
    
    def __init__(self, returns, p=1, q=1):
        self.returns = returns
        self.p = p
        self.q = q
        self.models = {}
        self.results = {}
        self.ic_scores = {}
        
    def fit_garch(self):
        """标准 GARCH(1,1) - 对称模型基准"""
        print("\n[1/3] 拟合 GARCH(1,1)...")
        try:
            # vol='GARCH' 是默认选项，这里显式写出来是为了代码可读性
            model = arch_model(self.returns, vol='Garch', p=self.p, q=self.q, dist='t')
            
            # 这个 tol 很关键：默认的 1e-6 在中国市场的某些极端时期会导致不收敛
            # 我们设成 1e-4 牺牲一点精度换取稳定性
            result = model.fit(disp='off', options={'ftol': 1e-4, 'maxiter': 500})
            
            self.models['GARCH'] = model
            self.results['GARCH'] = result
            self.ic_scores['GARCH'] = {'AIC': result.aic, 'BIC': result.bic}
            print(f"   AIC={result.aic:.2f}, BIC={result.bic:.2f}")
            
        except Exception as e:
            print(f"   ✗ GARCH 拟合失败: {str(e)}")
            self.ic_scores['GARCH'] = {'AIC': np.inf, 'BIC': np.inf}
    
    def fit_egarch(self):
        """EGARCH - 对数波动率模型（Nelson 1991）"""
        print("\n[2/3] 拟合 EGARCH(1,1)...")
        try:
            model = arch_model(self.returns, vol='EGARCH', p=self.p, q=self.q, dist='t')
            result = model.fit(disp='off', options={'ftol': 1e-4, 'maxiter': 500})
            
            self.models['EGARCH'] = model
            self.results['EGARCH'] = result
            self.ic_scores['EGARCH'] = {'AIC': result.aic, 'BIC': result.bic}
            print(f"   AIC={result.aic:.2f}, BIC={result.bic:.2f}")
            
            # EGARCH 的关键参数：gamma（不对称系数）
            # gamma < 0 意味着负冲击（利差扩大）会放大波动率
            if 'gamma[1]' in result.params:
                gamma = result.params['gamma[1]']
                print(f"   非对称系数 γ = {gamma:.4f} {'(负冲击放大波动)' if gamma < 0 else '(正冲击放大波动)'}")
            
        except Exception as e:
            print(f"   ✗ EGARCH 拟合失败: {str(e)}")
            self.ic_scores['EGARCH'] = {'AIC': np.inf, 'BIC': np.inf}
    
    def fit_gjr_garch(self):
        """GJR-GARCH - 阈值模型（Glosten, Jagannathan, Runkle 1993）"""
        print("\n[3/3] 拟合 GJR-GARCH(1,1)...")
        try:
            # vol='GARCH' + o=1 就是 GJR-GARCH
            # o 代表 asymmetric term (threshold effect)
            model = arch_model(self.returns, vol='Garch', p=self.p, o=1, q=self.q, dist='t')
            result = model.fit(disp='off', options={'ftol': 1e-4, 'maxiter': 500})
            
            self.models['GJR-GARCH'] = model
            self.results['GJR-GARCH'] = result
            self.ic_scores['GJR-GARCH'] = {'AIC': result.aic, 'BIC': result.bic}
            print(f"   AIC={result.aic:.2f}, BIC={result.bic:.2f}")
            
        except Exception as e:
            print(f"   ✗ GJR-GARCH 拟合失败: {str(e)}")
            self.ic_scores['GJR-GARCH'] = {'AIC': np.inf, 'BIC': np.inf}
    
    def run_tournament(self):
        """执行锦标赛 - 拟合所有模型并选出 Winner"""
        print("\n" + "="*60)
        print("开始 GARCH 模型锦标赛")
        print("="*60)
        
        self.fit_garch()
        self.fit_egarch()
        self.fit_gjr_garch()
        
        # 根据 AIC 选出最佳模型（AIC 越小越好）
        # 为什么用 AIC 而不是 BIC？BIC 对参数数量惩罚更重，可能过于保守
        winner = min(self.ic_scores, key=lambda x: self.ic_scores[x]['AIC'])
        
        print("\n" + "="*60)
        print(f"🏆 锦标赛获胜者: {winner}")
        print(f"   AIC = {self.ic_scores[winner]['AIC']:.2f}")
        print(f"   BIC = {self.ic_scores[winner]['BIC']:.2f}")
        print("="*60)
        
        return winner
    
    def get_conditional_volatility(self, model_name):
        """提取条件波动率序列"""
        if model_name not in self.results:
            raise ValueError(f"模型 {model_name} 未拟合")
        
        # conditional_volatility 是 GARCH 模型的核心输出
        # 它告诉我们「在给定历史信息下，今天的预期波动率是多少」
        return self.results[model_name].conditional_volatility

In [ ]:
# 执行 GARCH 锦标赛
vol_modeler = VolatilityModeler(returns, p=CONFIG['GARCH_P'], q=CONFIG['GARCH_Q'])
winner_model = vol_modeler.run_tournament()
winner_volatility = vol_modeler.get_conditional_volatility(winner_model)

---
## 第二部分: 模块 B - 卡尔曼滤波器 (Kalman Filter)
---

### 为什么需要 Kalman Filter？

市场报价的利差 = 真实的信用风险 + 流动性噪音 + 微观结构噪音

我们想要的是第一项，但直接观测到的是三者之和。

Kalman Filter 通过**状态空间模型**把信号和噪音分离：

- **状态方程**: 真实利差如何演化（随机游走假设）
- **观测方程**: 我们看到的报价 = 真实值 + 噪音

输出的 **Smoothed State** 就是我们估计的「基本面利差」。


In [ ]:
class KalmanSignalExtractor:
    """
    卡尔曼滤波器 - 从噪音中提取真实信号
    
    技术实现:
    - 使用 statsmodels 的 SARIMAX，令 order=(0,1,0) 就是 Local Level Model
    - 如果 Kalman 优化失败（在极短样本或极端波动下可能发生），fallback 到 60 日移动均值
    """
    
    def __init__(self, spread_series):
        self.spread = spread_series
        self.smoothed_state = None
        self.success = False
        
    def fit(self):
        """
        拟合 Local Level Model (随机游走 + 噪音)
        
        数学表达:
        - μ_t = μ_{t-1} + η_t        (状态方程: 随机游走)
        - y_t = μ_t + ε_t            (观测方程: 真实值 + 噪音)
        
        其中 η_t ~ N(0, σ_η^2), ε_t ~ N(0, σ_ε^2) 由 MLE 估计
        """
        print("\n" + "="*60)
        print("拟合卡尔曼滤波器 (Local Level Model)")
        print("="*60)
        
        try:
            # SARIMAX(order=(0,1,0)) 等价于:
            # - 0 个 AR 项
            # - 1 阶差分（这会自动构建随机游走状态空间）
            # - 0 个 MA 项
            model = SARIMAX(self.spread, order=(0, 1, 0), trend=None)
            
            # method='bfgs' 通常比默认的 'lbfgs' 更稳定
            # maxiter=500 给优化器足够多的迭代次数
            result = model.fit(disp=False, method='bfgs', maxiter=500)
            
            # get_smoothed_decomposition() 是 Kalman Smoother 的输出
            # 它用全部样本信息（包括未来）来估计每个时点的状态 → 比 Filter 更准确
            self.smoothed_state = result.smoothed_state[0]  # 取第一个状态变量
            self.success = True
            
            print("✓ Kalman Filter 拟合成功")
            print(f"  观测噪音标准差 σ_ε ≈ {np.sqrt(result.params['sigma2']):.2f} bps")
            
        except Exception as e:
            print(f"⚠️  Kalman Filter 优化失败: {str(e)}")
            print("   启用 Fallback: 使用 60 日滚动均值代替")
            
            # Fallback: 简单移动平均
            # 60 天窗口的选择：对应一个季度的财报周期
            # 太短（如 20 天）会追踪噪音，太长（如 120 天）反应太慢
            self.smoothed_state = self.spread.rolling(window=60, min_periods=1).mean()
            self.success = False
        
        return self.smoothed_state
    
    def get_signal_deviation(self):
        """
        计算当前利差偏离 Kalman 趋势的程度（标准化）
        
        这个指标可以用来构建均值回归策略:
        - deviation > +1.5σ → 利差高估，做空信号（预期收窄）
        - deviation < -1.5σ → 利差低估，做多信号（预期扩大）
        """
        if self.smoothed_state is None:
            raise ValueError("请先调用 fit()")
        
        deviation = self.spread - self.smoothed_state
        std = deviation.std()
        normalized_deviation = deviation / std
        
        return normalized_deviation

In [ ]:
# 执行卡尔曼滤波
kalman = KalmanSignalExtractor(clean_data['spread'])
smoothed_spread = kalman.fit()
signal_deviation = kalman.get_signal_deviation()

print(f"\n当前偏离度: {signal_deviation.iloc[-1]:.2f} σ")
if signal_deviation.iloc[-1] > 1.5:
    print("📊 交易信号: 利差高于趋势，考虑做空（预期收窄）")
elif signal_deviation.iloc[-1] < -1.5:
    print("📊 交易信号: 利差低于趋势，考虑做多（预期扩大）")
else:
    print("📊 交易信号: 在正常区间内，观望")

---
## 第二部分: 模块 C - 极值理论 (Extreme Value Theory)
---

### 为什么标准 VaR 不够用？

传统 VaR 假设收益率服从正态分布 → 严重低估尾部风险！

**EVT 的核心思想**:
只用极端值的数据来拟合尾部分布（Generalized Pareto Distribution）

**POT 方法**（Peaks Over Threshold）:
1. 选一个阈值 u（如 95% 分位数）
2. 只看超过 u 的那些「极端损失」
3. 拟合 GPD，推断 99% VaR

这比用全样本拟合正态分布更准确。


In [ ]:
class EVTRiskAnalyzer:
    """
    极值理论风险分析器 - 量化尾部风险
    
    方法: Peaks Over Threshold (POT)
    分布: Generalized Pareto Distribution (GPD)
    """
    
    def __init__(self, returns, threshold_percentile=0.95, confidence=0.99):
        self.returns = returns
        self.threshold_percentile = threshold_percentile
        self.confidence = confidence
        self.threshold = None
        self.gpd_params = None
        self.var = None
        
    def fit_gpd(self):
        """
        拟合 GPD 到负尾部（损失）
        
        技术细节:
        - 我们关注利差扩大（正收益）的尾部风险
        - 选择 95% 分位数作为阈值是经验法则（McNeil & Frey 2000）
        - 太低（如 90%）会引入非极值数据，太高（如 99%）样本太少
        """
        print("\n" + "="*60)
        print("拟合极值理论 (EVT) 模型")
        print("="*60)
        
        # 定义阈值
        self.threshold = self.returns.quantile(self.threshold_percentile)
        
        # 提取超过阈值的极值（exceedances）
        exceedances = self.returns[self.returns > self.threshold] - self.threshold
        
        if len(exceedances) < 10:
            print(f"⚠️  警告: 超过阈值的样本太少 ({len(exceedances)} 个)，EVT 估计可能不稳定")
        
        print(f"阈值 u = {self.threshold:.2f} bps (第 {self.threshold_percentile*100:.0f} 百分位)")
        print(f"超过阈值的极值样本: {len(exceedances)} 个")
        
        try:
            # 拟合 GPD: 用 MLE 估计形状参数 ξ 和尺度参数 σ
            # ξ > 0: 重尾分布（典型的金融市场）
            # ξ = 0: 指数分布
            # ξ < 0: 轻尾分布（有上界，罕见）
            shape, loc, scale = stats.genpareto.fit(exceedances, floc=0)  # floc=0 固定位置参数为 0
            
            self.gpd_params = {'shape': shape, 'scale': scale}
            
            print(f"\nGPD 参数估计:")
            print(f"  形状参数 ξ = {shape:.4f}")
            print(f"  尺度参数 σ = {scale:.4f}")
            
            if shape > 0.5:
                print(f"  ⚠️  ξ > 0.5 表示极端重尾，方差可能不存在！")
            elif shape > 0:
                print(f"  ✓ ξ > 0 确认重尾特征（符合金融数据）")
            
        except Exception as e:
            print(f"✗ GPD 拟合失败: {str(e)}")
            # Fallback: 用经验分位数
            self.gpd_params = None
    
    def calculate_var(self):
        """
        计算 EVT-VaR（基于 GPD 的 Value at Risk）
        
        公式（POT 方法）:
        VaR_α = u + (σ/ξ) * [ ((1-α)/(1-u_percentile))^(-ξ) - 1 ]
        
        其中:
        - u: 阈值
        - σ, ξ: GPD 参数
        - α: 置信水平（如 0.99）
        """
        if self.gpd_params is None:
            # Fallback: 使用经验分位数
            self.var = self.returns.quantile(self.confidence)
            print(f"\n使用经验分位数: {self.confidence*100}% VaR = {self.var:.2f} bps")
            return self.var
        
        shape = self.gpd_params['shape']
        scale = self.gpd_params['scale']
        
        # 超过阈值的概率
        p_exceed = 1 - self.threshold_percentile
        
        # EVT-VaR 公式
        if abs(shape) < 1e-6:  # shape ≈ 0 时，用指数分布公式
            self.var = self.threshold + scale * np.log((1 - self.confidence) / p_exceed)
        else:
            self.var = self.threshold + (scale / shape) * (
                ((1 - self.confidence) / p_exceed) ** (-shape) - 1
            )
        
        print(f"\n" + "="*60)
        print(f"🎯 EVT-VaR ({self.confidence*100}% 置信水平)")
        print(f"   最大日损失预期: {self.var:.2f} bps")
        print(f"   解读: 在 100 个交易日中，最坏的那一天利差扩大不超过此值")
        print("="*60)
        
        return self.var
    
    def get_tail_index(self):
        """返回尾部指数（Heavy-tail Index）"""
        if self.gpd_params is None:
            return None
        # 尾部指数 = 1/ξ（ξ 越大，尾部越重）
        return 1 / self.gpd_params['shape'] if self.gpd_params['shape'] > 0 else np.inf

In [ ]:
# 执行 EVT 分析
evt_analyzer = EVTRiskAnalyzer(
    returns, 
    threshold_percentile=CONFIG['EVT_THRESHOLD_PERCENTILE'],
    confidence=CONFIG['VaR_CONFIDENCE']
)
evt_analyzer.fit_gpd()
evt_var = evt_analyzer.calculate_var()

tail_index = evt_analyzer.get_tail_index()
if tail_index:
    print(f"\n尾部指数 α = {tail_index:.2f}")
    if tail_index < 3:
        print("  ⚠️  α < 3 意味着四阶矩不存在（峰度无限大）")
    elif tail_index < 4:
        print("  ⚠️  α < 4 意味着峰度非常高（极端厚尾）")

---
## 第三部分: 交互式可视化 (Plotly Dashboard)
---

### 三图联动展示完整风险画像


In [ ]:
# ============================================================================
# 图表 1: 信号与趋势 (卡尔曼滤波 vs 原始利差)
# ============================================================================

# 识别交易信号点（偏离 > 1.5σ）
buy_signals = signal_deviation[signal_deviation < -1.5]
sell_signals = signal_deviation[signal_deviation > 1.5]

fig1 = go.Figure()

# 原始利差（灰色半透明）
fig1.add_trace(go.Scatter(
    x=clean_data.index,
    y=clean_data['spread'],
    mode='lines',
    name='原始利差',
    line=dict(color='lightgray', width=1),
    opacity=0.5
))

# Kalman 平滑利差（蓝色粗线）
fig1.add_trace(go.Scatter(
    x=smoothed_spread.index,
    y=smoothed_spread.values,
    mode='lines',
    name='卡尔曼趋势',
    line=dict(color='#1f77b4', width=2.5)
))

# 买入信号（绿色向上三角）
if len(buy_signals) > 0:
    fig1.add_trace(go.Scatter(
        x=buy_signals.index,
        y=clean_data.loc[buy_signals.index, 'spread'],
        mode='markers',
        name='买入信号 (低估)',
        marker=dict(symbol='triangle-up', size=10, color='green')
    ))

# 卖出信号（红色向下三角）
if len(sell_signals) > 0:
    fig1.add_trace(go.Scatter(
        x=sell_signals.index,
        y=clean_data.loc[sell_signals.index, 'spread'],
        mode='markers',
        name='卖出信号 (高估)',
        marker=dict(symbol='triangle-down', size=10, color='red')
    ))

fig1.update_layout(
    title='图表 1: 信号提取 - 卡尔曼滤波 vs 原始利差',
    xaxis_title='日期',
    yaxis_title='利差 (bps)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig1.show()

In [ ]:
# ============================================================================
# 图表 2: 波动率结构（锦标赛获胜模型）
# ============================================================================

# 计算波动率的 90% 分位数（高波动阈值）
vol_threshold = winner_volatility.quantile(0.90)
high_vol_periods = winner_volatility[winner_volatility > vol_threshold]

fig2 = go.Figure()

# 条件波动率曲线
fig2.add_trace(go.Scatter(
    x=winner_volatility.index,
    y=winner_volatility.values,
    mode='lines',
    name=f'{winner_model} 条件波动率',
    line=dict(color='purple', width=1.5),
    fill='tozeroy',
    fillcolor='rgba(128, 0, 128, 0.1)'
))

# 高波动阈值线（虚线）
fig2.add_hline(
    y=vol_threshold,
    line_dash='dash',
    line_color='red',
    annotation_text=f'危机模式阈值 (90%分位: {vol_threshold:.2f} bps)',
    annotation_position='right'
)

# 标记高波动期（红色点）
if len(high_vol_periods) > 0:
    fig2.add_trace(go.Scatter(
        x=high_vol_periods.index,
        y=high_vol_periods.values,
        mode='markers',
        name='高波动期 (危机模式)',
        marker=dict(color='red', size=6, symbol='diamond')
    ))

fig2.update_layout(
    title=f'图表 2: 波动率结构 - {winner_model} 模型',
    xaxis_title='日期',
    yaxis_title='条件波动率 (bps)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig2.show()

In [ ]:
# ============================================================================
# 图表 3: 尾部风险锥（收益率分布 + Student-t 拟合 + VaR）
# ============================================================================

fig3 = go.Figure()

# 直方图（实际收益率分布）
fig3.add_trace(go.Histogram(
    x=returns,
    nbinsx=50,
    name='实际分布',
    marker_color='lightblue',
    opacity=0.7,
    histnorm='probability density'
))

# 拟合 Student-t 分布曲线
# 为什么用 t 分布而不是正态分布？因为金融收益率有肥尾
df_fit, loc_fit, scale_fit = stats.t.fit(returns)
x_range = np.linspace(returns.min(), returns.max(), 200)
t_pdf = stats.t.pdf(x_range, df_fit, loc_fit, scale_fit)

fig3.add_trace(go.Scatter(
    x=x_range,
    y=t_pdf,
    mode='lines',
    name=f'Student-t 拟合 (df={df_fit:.1f})',
    line=dict(color='darkblue', width=2.5)
))

# 正态分布对比线（展示肥尾效应）
normal_pdf = stats.norm.pdf(x_range, returns.mean(), returns.std())
fig3.add_trace(go.Scatter(
    x=x_range,
    y=normal_pdf,
    mode='lines',
    name='正态分布对比',
    line=dict(color='orange', width=2, dash='dot'),
    opacity=0.6
))

# VaR 标记线（红色虚线）
fig3.add_vline(
    x=evt_var,
    line_dash='dash',
    line_color='red',
    line_width=3,
    annotation_text=f'99% EVT-VaR: {evt_var:.2f} bps',
    annotation_position='top'
)

# 也标记经验分位数（对比）
empirical_var = returns.quantile(CONFIG['VaR_CONFIDENCE'])
fig3.add_vline(
    x=empirical_var,
    line_dash='dot',
    line_color='gray',
    annotation_text=f'经验99%分位: {empirical_var:.2f} bps',
    annotation_position='bottom'
)

fig3.update_layout(
    title='图表 3: 尾部风险锥 - 收益率分布与 VaR',
    xaxis_title='利差变化 (bps)',
    yaxis_title='概率密度',
    hovermode='x',
    template='plotly_white',
    height=500,
    showlegend=True
)

fig3.show()

print(f"\n对比分析:")
print(f"  EVT-VaR (99%):    {evt_var:.2f} bps")
print(f"  经验分位数 (99%): {empirical_var:.2f} bps")
print(f"  差异:             {evt_var - empirical_var:.2f} bps")
if evt_var > empirical_var:
    print(f"  ⚠️  EVT 估计的风险更高（更保守），这是肥尾效应的体现")
else:
    print(f"  ℹ️  EVT 与经验分位数接近，尾部风险可控")

---
## 第四部分: 战略输出 - "So What?" 报告
---


In [ ]:
# ============================================================================
# 生成战略分析报告
# ============================================================================

print("\n" + "="*80)
print(" " * 20 + "中国地方债利差战略分析报告")
print("="*80)

# ---------- 第一部分: 模型选择结果 ----------
print("\n【一、模型锦标赛结果】")
print("-" * 80)
print(f"✓ 获胜模型: {winner_model}")
print(f"  - AIC: {vol_modeler.ic_scores[winner_model]['AIC']:.2f}")
print(f"  - BIC: {vol_modeler.ic_scores[winner_model]['BIC']:.2f}")

# 对比所有模型
print("\n  模型对比:")
for model_name, scores in vol_modeler.ic_scores.items():
    winner_mark = "← 🏆" if model_name == winner_model else ""
    print(f"    {model_name:15s}  AIC={scores['AIC']:8.2f}  BIC={scores['BIC']:8.2f}  {winner_mark}")

print(f"\n  📊 结论: 基于 AIC 准则，{winner_model} 模型最能解释中国地方债利差的动态特征")

# ---------- 第二部分: 不对称效应检验 ----------
print("\n【二、波动率不对称效应】")
print("-" * 80)

asymmetry_detected = False
if winner_model == 'EGARCH':
    gamma = vol_modeler.results['EGARCH'].params.get('gamma[1]', 0)
    if gamma < -0.05:  # 显著负值
        print(f"  ✓ 检测到显著的不对称效应 (γ = {gamma:.4f})")
        print(f"    → 利差扩大（坏消息）引发的波动率上升幅度 > 利差收窄（好消息）")
        print(f"    → 这符合信用市场的经验：投资者对负面信息反应更剧烈")
        asymmetry_detected = True
    elif gamma > 0.05:
        print(f"  ⚠️  检测到反常的正不对称 (γ = {gamma:.4f})")
        print(f"    → 这在信用利差中较罕见,可能反映特殊的市场结构")
    else:
        print(f"  ℹ️  不对称效应不显著 (γ ≈ 0)")
        
elif winner_model == 'GJR-GARCH':
    gamma = vol_modeler.results['GJR-GARCH'].params.get('gamma[1]', 0)
    if gamma > 0.05:
        print(f"  ✓ 检测到显著的杠杆效应 (γ = {gamma:.4f})")
        print(f"    → 负冲击对波动率的影响比正冲击大 {gamma:.1%}")
        asymmetry_detected = True
    else:
        print(f"  ℹ️  杠杆效应不显著")
else:
    print(f"  ℹ️  标准 GARCH 模型胜出，未检测到不对称效应")
    print(f"    → 说明正负冲击对波动率的影响较为对称")

if asymmetry_detected:
    print("\n  🎯 交易含义: 在利差快速扩大时,应预期波动率会超比例上升,需加大对冲力度")

# ---------- 第三部分: 风险预警 ----------
print("\n【三、当前风险状况】")
print("-" * 80)

current_spread = clean_data['spread'].iloc[-1]
current_trend = smoothed_spread.iloc[-1]
deviation_bps = current_spread - current_trend
deviation_sigma = signal_deviation.iloc[-1]
current_vol = winner_volatility.iloc[-1]

print(f"  当前利差水平:        {current_spread:.2f} bps")
print(f"  卡尔曼趋势水平:      {current_trend:.2f} bps")
print(f"  偏离程度:            {deviation_bps:+.2f} bps ({deviation_sigma:+.2f}σ)")
print(f"  当前波动率:          {current_vol:.2f} bps/日")
print(f"  99% EVT-VaR:        {evt_var:.2f} bps (单日最大风险)")

# 风险等级判定
print("\n  风险等级判定:")
if abs(deviation_sigma) > 2.0:
    risk_level = "⚠️ 高风险"
    print(f"    {risk_level}: 利差严重偏离趋势 (>2σ)")
    if deviation_sigma > 0:
        print(f"      → 利差可能被高估,存在较大均值回归压力")
    else:
        print(f"      → 利差可能被低估,警惕信用事件风险")
elif abs(deviation_sigma) > 1.5:
    risk_level = "⚡ 中等风险"
    print(f"    {risk_level}: 利差偏离趋势 (1.5σ - 2σ)")
    print(f"      → 可考虑建立方向性仓位")
else:
    risk_level = "✓ 低风险"
    print(f"    {risk_level}: 利差在正常波动范围内 (<1.5σ)")
    print(f"      → 维持中性仓位")

# 波动率状态
vol_percentile = (winner_volatility < current_vol).mean()
print(f"\n  波动率状态: 当前处于历史 {vol_percentile:.1%} 分位")
if vol_percentile > 0.90:
    print(f"    ⚠️  危机模式: 波动率处于极高水平 (>90%分位)")
    print(f"      → 建议降低杠杆,增加现金储备")
elif vol_percentile > 0.75:
    print(f"    ⚡ 高波动期: 市场不确定性上升")
    print(f"      → 谨慎加仓,控制风险敞口")
else:
    print(f"    ✓ 正常波动期")

# ---------- 第四部分: 行动建议 ----------
print("\n【四、行动建议】")
print("-" * 80)

print("  基于当前分析,建议:")
print()

# 方向性建议
if deviation_sigma > 1.5:
    print("  1. 方向性策略: 🔴 做空利差 (预期收窄)")
    print(f"     - 入场点: {current_spread:.2f} bps")
    print(f"     - 目标价: {current_trend:.2f} bps (回归趋势)")
    print(f"     - 止损点: {current_spread + evt_var:.2f} bps (当前+VaR)")
elif deviation_sigma < -1.5:
    print("  1. 方向性策略: 🟢 做多利差 (预期扩大)")
    print(f"     - 入场点: {current_spread:.2f} bps")
    print(f"     - 目标价: {current_trend:.2f} bps (回归趋势)")
    print(f"     - 止损点: {max(0, current_spread - evt_var):.2f} bps (当前-VaR)")
else:
    print("  1. 方向性策略: ⚪ 中性观望")
    print(f"     - 当前利差在合理区间,等待更明确信号")

# 风险管理建议
print(f"\n  2. 风险管理:")
print(f"     - 单日 VaR 限额: {evt_var:.2f} bps")
print(f"     - 建议仓位规模: 假设风险预算为 R bps,则最大名义敞口 = R / {evt_var:.2f}")
if vol_percentile > 0.90:
    print(f"     - ⚠️  当前高波动环境,建议将仓位削减至正常水平的 50%-70%")

# 监控指标
print(f"\n  3. 关键监控指标:")
print(f"     - 偏离度: 当前 {deviation_sigma:+.2f}σ → 警戒线 ±1.5σ, 止损线 ±2.5σ")
print(f"     - 波动率: 当前 {current_vol:.2f} → 上升 20% 以上需重新评估风险敞口")
print(f"     - 趋势: 若卡尔曼趋势突破 {current_trend + current_vol*2:.2f} bps,说明市场regime切换")

print("\n" + "="*80)
print(" " * 30 + "报告完成")
print("="*80)

---
## 附录: 技术说明与局限性
---

### 模型假设

1. **GARCH 模型族**:
   - 假设条件方差可预测（基于历史波动率）
   - 局限性: 不能捕捉结构性突变（如政策急转弯）

2. **卡尔曼滤波器**:
   - 假设状态过程为线性高斯（实际市场可能非线性）
   - 局限性: 在极端尾部事件下估计不稳定

3. **极值理论**:
   - 假设尾部分布遵循 GPD（需要足够多的极值样本）
   - 局限性: 对阈值选择敏感,需要定期重新校准

### 数据要求

- **最小样本量**: 建议至少 500 个交易日（约 2 年）
- **数据频率**: 日频数据为佳（周频或月频会损失信息）
- **数据质量**: Wind EDB 数据需定期检查异常值

### 风险提示

⚠️ **本框架为研究工具,不构成投资建议**  
⚠️ **所有模型都是对现实的简化,实际交易需结合基本面和市场微观结构分析**  
⚠️ **历史统计特征不保证未来延续,特别是在政策环境变化时**  

---

### 代码复用说明

要将此框架应用到其他资产（如企业债利差、国债期限利差等）,只需修改:

```python
CONFIG = {
    'SOURCE': 'WIND_EDB',
    'TICKER': 'YOUR_TICKER_HERE',  # ← 改这里
    # ... 其他参数保持不变
}
```

然后重新运行所有单元格即可。

---
